# Tiny-ImageNet Training Notebook (Kaggle)

Train FracBNN / Ada-FracBNN on Tiny-ImageNet with the ImageNet training path in this repo.

| model\_id | Architecture | Description |
|-----------|-------------|-------------|
| 0 | `binput-pg-quant-shortcut` | Baseline PG |
| 1 | `adaptive-pg` | Adaptive PG (Ada-FracBNN) |
| 2 | `adaptive-pg-kd` | Adaptive PG + Knowledge Distillation |

## Before you start
1. **Add the dataset**: Click **Add data** and attach either `xiataokang/tinyimagenettorch` or a standard `tiny-imagenet-200` Kaggle dataset.
2. **Enable GPU**: Settings > Accelerator > **GPU T4 x2** (or P100).
3. **Enable Internet**: Settings > Internet > **On** (needed for git clone and pip install).
4. **Persistence**: Set Settings > Persistence > **Files only** so `/kaggle/working/` survives restarts.

### Important note
Tiny-ImageNet has **200 classes**, not 1000. This notebook uses the same `imagenet.py` pipeline, but the model head is adapted automatically to the dataset class count. Results are useful for validating the training pipeline and comparing the new changes, but they are **not** directly comparable to full ImageNet-1K numbers from the paper.

### Kaggle paths
- `/kaggle/input/` -- read-only attached datasets
- `/kaggle/working/` -- writable output directory; checkpoints and prepared Tiny-ImageNet data go here
- Session limit: 12 h with GPU

## Cell 1 -- GPU and Disk Check

In [ ]:
!nvidia-smi
import torch, os
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"VRAM: {mem_gb:.1f} GB")
    print(f"GPU count: {torch.cuda.device_count()}")
!df -h /kaggle/working

## Cell 2 -- Clone Repo

In [ ]:
import os

REPO_DIR = "/kaggle/working/endingengineering"
REPO_URL = "https://github.com/YOUR_USERNAME/endingengineering.git"  # <-- replace
BRANCH   = "feat/imagenet-colab-bootstrap"

if not os.path.exists(REPO_DIR):
    !git clone --branch $BRANCH $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git fetch origin $BRANCH && git checkout $BRANCH && git pull origin $BRANCH

os.chdir(REPO_DIR)
!git log --oneline -5

## Cell 3 -- Install Dependencies

In [ ]:
!pip install -q -r requirements.txt

## Cell 4 -- Smoke Test (verify ImageNet models load)

In [ ]:
!python test_imagenet_models.py

## Cell 5 -- Prepare Tiny-ImageNet Data

This notebook supports two Kaggle Tiny-ImageNet layouts:
1. a PyTorch-friendly dataset that already has `train/` and `val/` class folders
2. the standard raw `tiny-imagenet-200/` layout, where `val/images/` is flat and labels live in `val/val_annotations.txt`

The prep cell auto-detects the dataset root under `/kaggle/input/`, prepares a clean `train/` and `val/` layout under `/kaggle/working/tiny-imagenet/`, and makes it ready for `imagenet.py`. Train is usually symlinked directly; validation is reorganized only when needed.

In [ ]:
import os, pathlib, shutil

PREFERRED_SLUGS = [
    "tinyimagenettorch",
    "tiny-imagenet-200",
    "TinyImageNet200",
    "tinyimagenet200",
]
KNOWN_DATASET_PATHS = [
    pathlib.Path("/kaggle/input/datasets/nikhilshingadiya/tinyimagenet200"),
    pathlib.Path("/kaggle/input/tinyimagenet200"),
]
KAGGLE_INPUT_ROOT = pathlib.Path("/kaggle/input")
IMAGENET_DIR = pathlib.Path("/kaggle/working/tiny-imagenet")


def count_class_dirs(path: pathlib.Path) -> int:
    try:
        return sum(1 for p in path.iterdir() if p.is_dir())
    except (FileNotFoundError, NotADirectoryError, PermissionError):
        return 0


def looks_like_imagefolder_split(path: pathlib.Path, min_classes: int = 50) -> bool:
    return path.is_dir() and count_class_dirs(path) >= min_classes


def candidate_roots(base: pathlib.Path, max_depth: int = 4):
    roots = []
    queue = [(base, 0)]
    seen = set()
    while queue:
        current, depth = queue.pop(0)
        if current in seen or depth > max_depth:
            continue
        seen.add(current)
        roots.append(current)
        if depth == max_depth:
            continue
        try:
            children = [p for p in current.iterdir() if p.is_dir()]
        except (FileNotFoundError, NotADirectoryError, PermissionError):
            continue
        for child in sorted(children):
            queue.append((child, depth + 1))
    return roots


def discover_dataset_root():
    for known in KNOWN_DATASET_PATHS:
        if known.is_dir():
            print(f"Found known dataset path: {known}")
            return known

    attached_roots = sorted([p for p in KAGGLE_INPUT_ROOT.iterdir() if p.is_dir()])
    print("Attached Kaggle datasets:")
    for p in attached_roots:
        print(f"  {p.name}")

    for root in candidate_roots(KAGGLE_INPUT_ROOT, max_depth=3):
        if root.name in PREFERRED_SLUGS:
            return root
    return KAGGLE_INPUT_ROOT


def ensure_symlink_or_empty_dir(split_dst: pathlib.Path, split_src: pathlib.Path, split_name: str):
    if split_dst.is_symlink():
        linked_to = pathlib.Path(os.readlink(split_dst))
        if linked_to != split_src:
            split_dst.unlink()
            split_dst.symlink_to(split_src)
            print(f"Updated {split_name} symlink -> {split_src}")
        else:
            print(f"{split_name} symlink already points to {split_src}")
    elif split_dst.exists():
        if split_dst.is_dir() and count_class_dirs(split_dst) == 0:
            split_dst.rmdir()
            split_dst.symlink_to(split_src)
            print(f"Replaced empty {split_name}/ dir with symlink -> {split_src}")
        else:
            print(f"{split_name}/ already exists at {split_dst}")
    else:
        split_dst.symlink_to(split_src)
        print(f"Symlinked {split_name} -> {split_src}")


if not KAGGLE_INPUT_ROOT.exists():
    raise FileNotFoundError("/kaggle/input not found. Run this only inside a Kaggle notebook.")

search_root = discover_dataset_root()
print(f"\nUsing dataset search root: {search_root}")

raw_root = None
preformatted_train = None
preformatted_val = None

for candidate in candidate_roots(search_root, max_depth=4):
    train_dir = candidate / "train"
    val_dir = candidate / "val"
    raw_val_images = val_dir / "images"
    raw_val_annotations = val_dir / "val_annotations.txt"

    if looks_like_imagefolder_split(train_dir) and looks_like_imagefolder_split(val_dir):
        preformatted_train = train_dir
        preformatted_val = val_dir
        break

    if looks_like_imagefolder_split(train_dir) and raw_val_images.is_dir() and raw_val_annotations.exists():
        raw_root = candidate
        break

if preformatted_train is not None and preformatted_val is not None:
    print(f"\nFound PyTorch/ImageFolder-ready dataset:")
    print(f"  train: {preformatted_train}")
    print(f"  val:   {preformatted_val}")

    IMAGENET_DIR.mkdir(parents=True, exist_ok=True)
    ensure_symlink_or_empty_dir(IMAGENET_DIR / "train", preformatted_train, "train")
    ensure_symlink_or_empty_dir(IMAGENET_DIR / "val", preformatted_val, "val")
else:
    if raw_root is None:
        raise FileNotFoundError(
            "Could not find a Tiny-ImageNet dataset layout. Attach a Kaggle Tiny-ImageNet dataset "
            "such as `xiataokang/tinyimagenettorch` or `tiny-imagenet-200` and rerun this cell."
        )

    print(f"\nFound standard Tiny-ImageNet raw layout: {raw_root}")
    train_src = raw_root / "train"
    val_images = raw_root / "val" / "images"
    val_annotations = raw_root / "val" / "val_annotations.txt"

    IMAGENET_DIR.mkdir(parents=True, exist_ok=True)
    ensure_symlink_or_empty_dir(IMAGENET_DIR / "train", train_src, "train")

    val_dst = IMAGENET_DIR / "val"
    marker = val_dst / ".reorganised"
    if marker.exists():
        print("val/ already reorganised -- skipping")
    else:
        if val_dst.exists() and not val_dst.is_symlink():
            shutil.rmtree(val_dst)
        val_dst.mkdir(parents=True, exist_ok=True)

        moved = 0
        with open(val_annotations) as f:
            for line in f:
                image_name, wnid = line.split("\t", 2)[:2]
                cls_dir = val_dst / wnid
                cls_dir.mkdir(exist_ok=True)
                src = val_images / image_name
                dst = cls_dir / image_name
                if src.exists() and not dst.exists():
                    shutil.copy2(src, dst)
                    moved += 1
        marker.touch()
        print(f"Reorganised validation set into class folders ({moved} images copied)")

train_classes = count_class_dirs(IMAGENET_DIR / "train")
val_classes = count_class_dirs(IMAGENET_DIR / "val")
print(f"\nTrain classes: {train_classes}, Val classes: {val_classes}")
!du -sh /kaggle/working/tiny-imagenet

## Cell 6 -- Set Checkpoint Directory

Everything under `/kaggle/working/` is saved as notebook output and persists across sessions
(when Persistence is set to **Files only**).

In [ ]:
SAVE_DIR = "/kaggle/working/checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Checkpoints will be saved to {SAVE_DIR}")

---
## Cell 7 -- Train Model 0: Baseline PG (`binput-pg-quant-shortcut`)

For Tiny-ImageNet, start with baseline or adaptive PG first.  
A good first run is `30-100` epochs depending on your GPU budget.  
Kaggle GPU sessions last 12 h -- use Cell 11 to resume across sessions if needed.

In [ ]:
os.chdir(REPO_DIR)

!CUDA_VISIBLE_DEVICES=0,1 python imagenet.py \
    -id 0 \
    -e  100 \
    -b  64 \
    -lr 5e-4 \
    -wd 1e-5 \
    -w  2 \
    -d  $IMAGENET_DIR \
    --label_smoothing 0.1 \
    -gpu 0,1 \
    -s

!cp -a save_ImageNet_model "$SAVE_DIR/save_ImageNet_model_baseline"

## Cell 8 -- Train Model 1: Adaptive PG (`adaptive-pg`)

In [ ]:
!CUDA_VISIBLE_DEVICES=0,1 python imagenet.py \
    -id 1 \
    -e  100 \
    -b  64 \
    -lr 5e-4 \
    -wd 1e-5 \
    -w  2 \
    -d  $IMAGENET_DIR \
    -ts 0.15 \
    -sw 0.01 \
    --label_smoothing 0.1 \
    -gpu 0,1 \
    -s

!cp -a save_ImageNet_model "$SAVE_DIR/save_ImageNet_model_adaptive"

## Cell 9 -- Train Model 2: Adaptive PG + KD (`adaptive-pg-kd`)\n\nFor Tiny-ImageNet, `--teacher_pretrained` from torchvision will **not** work because those weights are only for 1000-class ImageNet.\nUse this cell only if you already have a Tiny-ImageNet teacher checkpoint trained on the same 200 classes.

In [ ]:
TEACHER_PATH = "/kaggle/working/tiny_imagenet_teacher.pt"  # set this only if you have a Tiny-ImageNet teacher

if os.path.exists(TEACHER_PATH):
    !CUDA_VISIBLE_DEVICES=0,1 python imagenet.py \
        -id 2 \
        -e  100 \
        -b  32 \
        -lr 5e-4 \
        -wd 1e-5 \
        -w  2 \
        -d  $IMAGENET_DIR \
        -ts 0.15 \
        -sw 0.01 \
        -temp  4.0 \
        -alpha 0.7 \
        --teacher_arch resnet50 \
        -tp $TEACHER_PATH \
        --label_smoothing 0.1 \
        -gpu 0,1 \
        -s

    !cp -a save_ImageNet_model "$SAVE_DIR/save_ImageNet_model_kd"
else:
    print("Skipping KD run: set TEACHER_PATH to a Tiny-ImageNet teacher checkpoint first.")

## Cell 10 -- Evaluate All Saved Models

In [ ]:
import glob

variants = [
    ("Baseline PG",      0, f"{SAVE_DIR}/save_ImageNet_model_baseline"),
    ("Adaptive PG",      1, f"{SAVE_DIR}/save_ImageNet_model_adaptive"),
    ("Adaptive PG + KD", 2, f"{SAVE_DIR}/save_ImageNet_model_kd"),
]

for name, mid, path in variants:
    ckpts = sorted(glob.glob(os.path.join(path, "model_*.pt")))
    if not ckpts:
        print(f"\n[SKIP] {name} -- no checkpoint found in {path}")
        continue
    ckpt = ckpts[0]
    print(f"\n{'='*60}")
    print(f"Evaluating: {name}  (checkpoint: {ckpt})")
    print(f"{'='*60}")
    !CUDA_VISIBLE_DEVICES=0,1 python imagenet.py -id $mid -t -r "$ckpt" -d $IMAGENET_DIR -b 128 -gpu 0,1

## Cell 11 -- Resume Training After Session Timeout

Kaggle GPU sessions last up to 12 h.  
Re-run Cells 1-6 (fast -- repo is cached, Tiny-ImageNet prep is already done), then use this cell.  
Edit the three variables below to match the variant you were training.

In [ ]:
# --- edit these three lines to match your variant ---
MODEL_ID   = 1
CKPT_MODEL = f"{SAVE_DIR}/save_ImageNet_model_adaptive/model_adaptive-pg.pt"
CKPT_STATE = f"{SAVE_DIR}/save_ImageNet_model_adaptive/states_adaptive-pg.pt"
# ----------------------------------------------------

os.chdir(REPO_DIR)

!CUDA_VISIBLE_DEVICES=0,1 python imagenet.py \
    -id $MODEL_ID \
    -e  100 \
    -b  64 \
    -lr 5e-4 \
    -wd 1e-5 \
    -w  2 \
    -d  $IMAGENET_DIR \
    -r  "$CKPT_MODEL" \
    -l  "$CKPT_STATE" \
    -ts 0.15 \
    -sw 0.01 \
    --label_smoothing 0.1 \
    -gpu 0,1 \
    -s

---
## Reference

### Hyperparameters for Tiny-ImageNet

| Parameter | Value | Notes |
|-----------|-------|-------|
| Epochs | 100 | Good starting schedule for Tiny-ImageNet |
| GPUs | 2 (`0,1`) | Uses `nn.DataParallel` in `imagenet.py` |
| Batch size | 128 (baseline/adaptive), 64 (KD) | Divisible across 2 GPUs |
| Learning rate | 5e-4, linear decay | |
| Weight decay | 1e-5 | |
| Label smoothing | 0.1 | |
| Target sparsity | 0.15 | 15% channels get 2-bit |
| Sparsity weight | 0.01 | |
| KD temperature | 4.0 | |
| KD alpha | 0.7 | |
| Teacher | custom Tiny-ImageNet checkpoint only | torchvision pretrained teachers are 1000-class only |

### Accuracy note

Tiny-ImageNet is a useful step between CIFAR and full ImageNet. It is good for comparing:
- baseline vs adaptive PG
- whether the new pushed changes help
- whether the ImageNet training path behaves sensibly on a 200-class problem

These numbers are **not** directly comparable to full ImageNet-1K results from the paper.

### Kaggle runtime notes

- GPU quota: 30 h/week (T4 x2) or 20 h/week (P100)
- Session limit: 12 h
- Tiny-ImageNet is much smaller than full ImageNet, so each epoch should be substantially faster
- 100 epochs may still need multiple sessions; use Cell 11 to resume
- `/kaggle/working/` persists across sessions when Persistence is enabled